# D171 — SQL Set Operations

SQL queries return **sets of rows**. Set operators combine the results of two or more `SELECT` statements vertically.

This notebook uses a very small course-enrollment dataset so that every result can be checked by hand. It covers:

- `UNION` and `UNION ALL`;
- `INTERSECT` and `INTERSECT ALL`;
- `EXCEPT` and `EXCEPT ALL`;
- column-count and data-type compatibility;
- duplicate and `NULL` behavior;
- column names, `ORDER BY`, `LIMIT`, and operator precedence;
- union, intersection, difference, symmetric difference, and three-set questions; and
- the difference between set operators and joins.

`INTERSECT` and `EXCEPT` require MySQL 8.0.31 or newer. The notebook prints the server version before running those examples.


## 1. Connect to MySQL

The connection settings match the D16 notebooks. Environment variables can override the classroom defaults.


In [ ]:
import os
import mysql.connector
from mysql.connector import Error

connection = mysql.connector.connect(
    host=os.environ.get("MYSQL_HOSTNAME", "127.0.0.1"),
    port=int(os.environ.get("MYSQL_PORT", "3306")),
    user=os.environ.get("MYSQL_USERNAME", "root"),
    password=os.environ.get("MYSQL_PASSWORD", "root"),
    database=os.environ.get("MYSQL_DATABASE", "olist_import_lab"),
)

print("Connected:", connection.is_connected())
print("MySQL version:", connection.server_info)


## 2. Query helpers

`execute_sql` executes one SQL statement and prints a compact result table. The helper also supports table-creation and insert statements used by the example dataset.


In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return

    text_rows = [["NULL" if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    print(" | ".join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None, max_rows=50):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if cursor.with_rows:
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchmany(max_rows + 1)
            visible_rows = rows[:max_rows]
            print_rows(columns, visible_rows)
            if len(rows) > max_rows:
                print(f"... showing the first {max_rows} rows")
            return visible_rows

        connection.commit()
        print(f"Statement completed. Affected rows: {cursor.rowcount:,}")
        return cursor.rowcount
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()


## 3. Small example dataset

Three temporary tables represent enrollments in Python, SQL, and data-visualization courses. Temporary tables exist only in this connection and disappear when it closes.

Notice the deliberate overlaps:

| Student | Python | SQL | Visualization |
|---|:---:|:---:|:---:|
| 101 Asha | ✓ |  |  |
| 102 Ben | ✓ | ✓ |  |
| 103 Chen | ✓ | ✓ | ✓ |
| 104 Divya | ✓ |  | ✓ |
| 105 Elena |  | ✓ |  |
| 106 Farah |  | ✓ | ✓ |
| 107 Gopal |  |  | ✓ |
| 108 name unavailable | ✓ | ✓ |  |

The same `(student_id, student_name)` row in two course tables lets us see when duplicates are removed or preserved.


In [ ]:
for table_name in ("set_python_students", "set_sql_students", "set_viz_students"):
    execute_sql(f"""
    CREATE TEMPORARY TABLE IF NOT EXISTS {table_name} (
        student_id INT NOT NULL,
        student_name VARCHAR(50) NULL,
        city VARCHAR(30) NOT NULL
    )
    """)
    execute_sql(f"DELETE FROM {table_name}")

python_rows = [
    (101, "Asha", "Chennai"), (102, "Ben", "Bengaluru"),
    (103, "Chen", "Chennai"), (104, "Divya", "Pune"),
    (108, None, "Delhi"),
]
sql_rows = [
    (102, "Ben", "Bengaluru"), (103, "Chen", "Chennai"),
    (105, "Elena", "Mumbai"), (106, "Farah", "Pune"),
    (108, None, "Delhi"),
]
viz_rows = [
    (103, "Chen", "Chennai"), (104, "Divya", "Pune"),
    (106, "Farah", "Pune"), (107, "Gopal", "Kochi"),
]

insert_sql = "INSERT INTO {} (student_id, student_name, city) VALUES (%s, %s, %s)"
cursor = connection.cursor()
try:
    cursor.executemany(insert_sql.format("set_python_students"), python_rows)
    cursor.executemany(insert_sql.format("set_sql_students"), sql_rows)
    cursor.executemany(insert_sql.format("set_viz_students"), viz_rows)
    connection.commit()
finally:
    cursor.close()

print("Temporary example data is ready.")


In [ ]:
execute_sql("SELECT 'Python' AS course, student_id, student_name, city FROM set_python_students")
execute_sql("SELECT 'SQL' AS course, student_id, student_name, city FROM set_sql_students")
execute_sql("SELECT 'Visualization' AS course, student_id, student_name, city FROM set_viz_students")


## 4. Set-operation rules

Each `SELECT` is an input set. The inputs must follow these rules:

1. They must return the same number of columns.
2. Corresponding columns must have compatible data types.
3. Columns are matched by **position**, not by name.
4. Final column names come from the first `SELECT`.
5. A final `ORDER BY` applies to the combined result.

Set operators stack rows. A `JOIN`, in contrast, matches tables and normally adds columns horizontally.


## 5. `UNION`: combine and remove duplicates

`UNION` returns rows found in either input and performs an implicit distinct operation.

Expected: 8 students. Ben, Chen, and the unnamed student occur in both source tables but appear once in the result.


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
UNION
SELECT student_id, student_name FROM set_sql_students
ORDER BY student_id
""")


## 6. `UNION ALL`: combine and keep duplicates

`UNION ALL` keeps every input row. It avoids duplicate removal and is generally faster when deduplication is not required.

Expected: 10 rows. Student IDs 102, 103, and 108 each appear twice.


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
UNION ALL
SELECT student_id, student_name FROM set_sql_students
ORDER BY student_id
""")


### Make duplicate origins visible

Adding a course label changes the row. Ben in Python is no longer identical to Ben in SQL, so `UNION` keeps both rows.


In [ ]:
execute_sql("""
SELECT student_id, student_name, 'Python' AS course FROM set_python_students
UNION
SELECT student_id, student_name, 'SQL' AS course FROM set_sql_students
ORDER BY student_id, course
""")


## 7. `INTERSECT`: rows common to both inputs

`INTERSECT` returns distinct rows present in both results.

Expected: students 102, 103, and 108.


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
INTERSECT
SELECT student_id, student_name FROM set_sql_students
ORDER BY student_id
""")


## 8. `EXCEPT`: rows in the first input but not the second

Set difference is directional: `A EXCEPT B` is not the same as `B EXCEPT A`.

Expected for Python minus SQL: Asha (101) and Divya (104).


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
EXCEPT
SELECT student_id, student_name FROM set_sql_students
ORDER BY student_id
""")


Expected for SQL minus Python: Elena (105) and Farah (106).


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_sql_students
EXCEPT
SELECT student_id, student_name FROM set_python_students
ORDER BY student_id
""")


## 9. Symmetric difference: exactly one of two sets

The symmetric difference contains members in Python or SQL, but not both. It combines the two directional differences.

Expected: 101, 104, 105, and 106.


In [ ]:
execute_sql("""
(SELECT student_id, student_name FROM set_python_students
 EXCEPT
 SELECT student_id, student_name FROM set_sql_students)
UNION
(SELECT student_id, student_name FROM set_sql_students
 EXCEPT
 SELECT student_id, student_name FROM set_python_students)
ORDER BY student_id
""")


## 10. `ALL` variants and duplicate counts

`INTERSECT ALL` and `EXCEPT ALL` preserve duplicate multiplicity. For a row occurring `m` times on the left and `n` times on the right:

- `INTERSECT ALL` returns it `MIN(m, n)` times;
- `EXCEPT ALL` returns it `MAX(m - n, 0)` times.

The next example uses inline result sets so the repeated values are obvious.


In [ ]:
execute_sql("""
(SELECT 1 AS value UNION ALL SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 2)
INTERSECT ALL
(SELECT 1 AS value UNION ALL SELECT 1 UNION ALL SELECT 3)
ORDER BY value
""")


In [ ]:
execute_sql("""
(SELECT 1 AS value UNION ALL SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 2)
EXCEPT ALL
(SELECT 1 AS value UNION ALL SELECT 3)
ORDER BY value
""")


## 11. `NULL` in set operations

For duplicate elimination and set matching, two rows containing `NULL` in the same positions are treated as duplicates. This differs from an ordinary predicate such as `NULL = NULL`, whose result is unknown.

Expected: one `NULL` row and one `Known` row.


In [ ]:
execute_sql("""
SELECT NULL AS possible_name
UNION
SELECT NULL
UNION
SELECT 'Known'
""")


In [ ]:
execute_sql("""
SELECT NULL AS possible_name
INTERSECT
SELECT NULL
""")


## 12. Column position and output names

The alias from the first query becomes the result-column name. SQL aligns columns by position, so selecting `student_name, student_id` in one branch and `student_id, student_name` in another would be logically incorrect even if MySQL can convert the values.


In [ ]:
execute_sql("""
SELECT student_id AS learner_id, student_name AS learner_name
FROM set_python_students
UNION
SELECT student_id AS ignored_id_alias, student_name AS ignored_name_alias
FROM set_sql_students
ORDER BY learner_id
""")


## 13. Compatible data types

Corresponding columns should represent the same kind of fact and have compatible types. `CAST` can make the intended common type explicit.


In [ ]:
execute_sql("""
SELECT CAST(student_id AS CHAR) AS identifier, 'student' AS record_type
FROM set_python_students
UNION
SELECT city AS identifier, 'city' AS record_type
FROM set_sql_students
ORDER BY record_type, identifier
""")


## 14. `ORDER BY` and `LIMIT` on a combined result

Place the final `ORDER BY` after the last branch. It sorts the complete set. Use result-column names or positions exposed by the first query.

Expected: the three greatest student IDs across all courses.


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
UNION
SELECT student_id, student_name FROM set_sql_students
UNION
SELECT student_id, student_name FROM set_viz_students
ORDER BY student_id DESC
LIMIT 3
""")


### Limit an individual branch with parentheses

Parentheses allow a branch to have its own `ORDER BY` and `LIMIT`. This example takes the two smallest IDs from each course and keeps their course labels.


In [ ]:
execute_sql("""
(SELECT student_id, student_name, 'Python' AS course
 FROM set_python_students ORDER BY student_id LIMIT 2)
UNION ALL
(SELECT student_id, student_name, 'SQL' AS course
 FROM set_sql_students ORDER BY student_id LIMIT 2)
ORDER BY course, student_id
""")


## 15. Operator precedence

MySQL evaluates `INTERSECT` before `UNION` and `EXCEPT`. Parentheses make the intended grouping clear and are strongly recommended when operators are mixed.

The query below means: Python students who are also SQL students, then combine them with all visualization students.


In [ ]:
execute_sql("""
(SELECT student_id, student_name FROM set_python_students
 INTERSECT
 SELECT student_id, student_name FROM set_sql_students)
UNION
SELECT student_id, student_name FROM set_viz_students
ORDER BY student_id
""")


## 16. Across three sets

### Students enrolled in all three courses

Expected: Chen (103).


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
INTERSECT
SELECT student_id, student_name FROM set_sql_students
INTERSECT
SELECT student_id, student_name FROM set_viz_students
ORDER BY student_id
""")


### Students enrolled in at least one course

Expected: all eight distinct students.


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
UNION
SELECT student_id, student_name FROM set_sql_students
UNION
SELECT student_id, student_name FROM set_viz_students
ORDER BY student_id
""")


### Students in Python but not in either other course

First combine SQL and visualization, then subtract that combined set from Python. Expected: Asha (101).


In [ ]:
execute_sql("""
SELECT student_id, student_name FROM set_python_students
EXCEPT
(SELECT student_id, student_name FROM set_sql_students
 UNION
 SELECT student_id, student_name FROM set_viz_students)
ORDER BY student_id
""")


## 17. Set operators versus joins

Both can answer “common rows,” but they produce different shapes:

- `INTERSECT` compares complete projected rows and stacks results vertically.
- `INNER JOIN` matches using an `ON` condition and can return columns from both sides.

This join shows the cities recorded for students common to Python and SQL.


In [ ]:
execute_sql("""
SELECT p.student_id, p.student_name,
       p.city AS python_city, s.city AS sql_city
FROM set_python_students AS p
INNER JOIN set_sql_students AS s
    ON s.student_id = p.student_id
ORDER BY p.student_id
""")


## 18. Applying sets to the Olist data

The same operators work with the D16 tables. This example finds customer states that also contain sellers.


In [ ]:
execute_sql("""
SELECT customer_state AS state FROM olist_customers
INTERSECT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state
""")


This example finds customer states with no seller records. The direction of `EXCEPT` matters.


In [ ]:
execute_sql("""
SELECT customer_state AS state FROM olist_customers
EXCEPT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state
""")


## 19. Common mistakes

| Mistake | Why it is a problem | Fix |
|---|---|---|
| Different column counts | Set branches cannot align | Select the same number of columns |
| Unrelated column meanings | Positionally valid but logically wrong | Align equivalent business attributes |
| Using `UNION` automatically | It spends work removing duplicates | Use `UNION ALL` when duplicates are acceptable |
| Expecting `UNION ALL` to deduplicate | It preserves every row | Use `UNION` when a distinct set is required |
| Reversing `EXCEPT` inputs | Difference is directional | State the left and right sets in words first |
| Branch-level `ORDER BY` without parentheses | It does not define final set order | Put final ordering at the end or parenthesize a limited branch |
| Confusing a set operation with a join | Set operations add rows; joins usually add columns | Choose based on the required result shape |
| Omitting parentheses in mixed operations | Precedence may change the meaning | Group the intended operations explicitly |


## 20. Summary

| Requirement | Operator or pattern |
|---|---|
| Either A or B, distinct | `UNION` |
| Every row from A and B | `UNION ALL` |
| Both A and B | `INTERSECT` |
| A but not B | `EXCEPT` |
| A or B but not both | `(A EXCEPT B) UNION (B EXCEPT A)` |
| Preserve duplicate multiplicity | `INTERSECT ALL` / `EXCEPT ALL` |

Prefer `UNION ALL` when duplicates are acceptable or impossible. Use parentheses to make multi-operator intent obvious.


## 21. Close the connection

Closing the connection also removes the three temporary example tables.


In [ ]:
if connection.is_connected():
    connection.close()
print("MySQL connection closed.")
